# Wbudowany serwer HTTP - Interfejsy sieciowe

W tym notatniku:
- **Czym jest interfejs sieciowy?** (obrazowo, prostymi słowami)
- **Interfejs vs Port vs IP** - co to jest, czym się różnią
- **Nasłuchiwanie (bind)** - `127.0.0.1` vs `0.0.0.0` vs konkretny IP
- **FastAPI + uvicorn** - praktyczne przykłady uruchamiania
- **Kiedy używać którego?** - development, docker, produkcja

## Czym jest interfejs sieciowy? (obrazowo)

### Analogia: Blok mieszkalny z wieloma wejściami

Wyobraź sobie **blok mieszkalny** (Twój komputer):

```
┌─────────────────────────────────────────┐
│         BLOK MIESZKALNY (Komputer)      │
│                                         │
│  Wejście A (eth0)     Wejście B (wlan0) │
│  ul. Główna 10        ul. WiFi 20       │
│  ┌──────┐             ┌──────┐          │
│  │  🚪  │             │  🚪  │          │
│  └──────┘             └──────┘          │
│                                         │
│  Wejście C (lo - loopback)              │
│  ul. Wewnętrzna 127                     │
│  ┌──────┐                               │
│  │  🚪  │  <- tylko dla mieszkańców!    │
│  └──────┘                               │
│                                         │
│  W środku mieszkania (porty):           │
│  - Mieszkanie 80 (HTTP)                 │
│  - Mieszkanie 443 (HTTPS)               │
│  - Mieszkanie 8000 (FastAPI dev)        │
└─────────────────────────────────────────┘
```

### Kluczowe pojęcia:

1. **Interfejs sieciowy** (network interface) = **drzwi wejściowe do bloku**
   - `eth0` - kabel sieciowy (Wejście A)
   - `wlan0` - WiFi (Wejście B)
   - `lo` - loopback, wewnętrzne (Wejście C - tylko dla mieszkańców)
   - `docker0` - most Docker (Wejście D - dla kontenerów)

2. **Adres IP** = **adres ulicy dla danego wejścia**
   - `eth0` ma adres `192.168.1.10` (ul. Główna 10)
   - `wlan0` ma adres `192.168.1.20` (ul. WiFi 20)
   - `lo` ma adres `127.0.0.1` (ul. Wewnętrzna 127 - localhost)

3. **Port** = **numer mieszkania w bloku**
   - Port 80 - mieszkanie HTTP
   - Port 443 - mieszkanie HTTPS
   - Port 8000 - mieszkanie FastAPI

4. **Socket** = **pełny adres (ulica + mieszkanie)**
   - `192.168.1.10:8000` - ul. Główna 10, mieszkanie 8000
   - `127.0.0.1:8000` - ul. Wewnętrzna 127, mieszkanie 8000

### Ważne!

**Interfejs NIE JEST portem!**
- **Interfejs** (eth0, wlan0) - fizyczne/wirtualne "wejście" do komputera
- **Port** (80, 8000) - logiczny "numer mieszkania" wewnątrz komputera
- **Jeden komputer** może mieć wiele interfejsów (wiele wejść)
- **Jeden interfejs** może mieć wiele portów (wiele mieszkań)

## Terminologia - co można pomylić?

| Termin | Co to jest? | Przykład | Analogia |
|--------|-------------|----------|----------|
| **Network Interface** | Fizyczna/wirtualna "karta sieciowa" | `eth0`, `wlan0`, `lo` | Drzwi wejściowe do bloku |
| **IP Address** | Numer przypisany do interfejsu | `192.168.1.10`, `127.0.0.1` | Adres ulicy dla danego wejścia |
| **Port** | Numer logiczny (1-65535) | `80`, `443`, `8000` | Numer mieszkania w bloku |
| **Socket** | Kombinacja IP:Port | `192.168.1.10:8000` | Pełny adres (ulica + mieszkanie) |
| **Host** | Nazwa DNS lub IP | `localhost`, `example.com` | Nazwa budynku |
| **Localhost** | Zawsze `127.0.0.1` | `localhost` = `127.0.0.1` | "Wewnętrzne wejście" dla mieszkańców |
| **Loopback** | Interfejs wewnętrzny (`lo`) | `127.0.0.1` | Wejście tylko dla mieszkańców bloku |

### Przykład ilustrujący różnicę:

```bash
# Interfejs: eth0
# IP: 192.168.1.10
# Port: 8000
# Socket: 192.168.1.10:8000

# Pełny adres FastAPI:
http://192.168.1.10:8000
#      └─────┬─────┘ └┬─┘
#         IP (adres)  Port (mieszkanie)
```

### Częste błędne myślenie:

❌ **Błąd:** "Interfejs to port 8000"  
✅ **Prawidłowo:** "Interfejs to eth0, który ma IP 192.168.1.10, a aplikacja nasłuchuje na porcie 8000"

❌ **Błąd:** "127.0.0.1 to interfejs"  
✅ **Prawidłowo:** "`lo` (loopback) to interfejs, a `127.0.0.1` to jego adres IP"

## Jakie interfejsy ma Twój komputer?

Sprawdźmy listę interfejsów sieciowych na Twoim komputerze.

### Linux/macOS - polecenie `ip addr` lub `ifconfig`

In [ ]:
# Linux: ip addr (nowoczesne)
!ip addr show

# Krótka wersja - tylko nazwy interfejsów i IP
# !ip -br addr

### Przykładowe wyjście `ip addr`:

```
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536
    inet 127.0.0.1/8 scope host lo
    ↑    └──────┬──────┘
    │      Adres IP loopback (localhost)
    Interfejs: lo (loopback - wewnętrzny)

2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500
    inet 192.168.1.10/24 brd 192.168.1.255 scope global eth0
    ↑    └─────┬──────┘
    │      Adres IP interfejsu eth0 (kabel)
    Interfejs: eth0 (kabel sieciowy)

3: wlan0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500
    inet 192.168.1.20/24 brd 192.168.1.255 scope global wlan0
    ↑    └─────┬──────┘
    │      Adres IP interfejsu wlan0 (WiFi)
    Interfejs: wlan0 (WiFi)

4: docker0: <NO-CARRIER,BROADCAST,MULTICAST,UP> mtu 1500
    inet 172.17.0.1/16 brd 172.17.255.255 scope global docker0
    ↑    └─────┬────┘
    │      Adres IP interfejsu docker0 (most Docker)
    Interfejs: docker0 (wirtualny - dla kontenerów)
```

### Najważniejsze interfejsy:

- **`lo`** (loopback) - `127.0.0.1` - wewnętrzny, tylko localhost
- **`eth0`** (ethernet) - kabel sieciowy, np. `192.168.1.10`
- **`wlan0`** (wireless LAN) - WiFi, np. `192.168.1.20`
- **`docker0`** - wirtualny most dla kontenerów Docker, np. `172.17.0.1`

## Nasłuchiwanie (bind) - gdzie serwer przyjmuje połączenia?

Kiedy uruchamiasz serwer HTTP (FastAPI + uvicorn), musisz zdecydować **na którym interfejsie** będzie nasłuchiwał.

### Opcje nasłuchiwania:

| Host (--host) | Co to znaczy? | Kto może się połączyć? | Kiedy używać? |
|---------------|---------------|------------------------|---------------|
| **`127.0.0.1`** (localhost) | Tylko interfejs loopback (`lo`) | **Tylko Ty** na tym samym komputerze | Development (bezpieczeństwo) |
| **`0.0.0.0`** | **Wszystkie interfejsy** (`lo`, `eth0`, `wlan0`, `docker0`) | **Wszyscy** (localhost + sieć lokalna + internet) | Docker, produkcja z firewall |
| **Konkretny IP** (np. `192.168.1.10`) | Tylko interfejs z tym IP | **Tylko przez ten interfejs** | Ograniczenie do jednej sieci |

### Obrazowo:

```
┌─────────────────────────────────────────┐
│         BLOK MIESZKALNY (Komputer)      │
│                                         │
│  --host 127.0.0.1 (localhost)           │
│  Wejście C (lo) OTWARTE ✅              │
│  ┌──────┐                               │
│  │  🚪  │ <- tylko mieszkańcy!          │
│  └──────┘                               │
│  Wejście A (eth0) ZAMKNIĘTE ❌          │
│  Wejście B (wlan0) ZAMKNIĘTE ❌         │
│                                         │
│  --host 0.0.0.0 (wszystkie)             │
│  Wejście A (eth0) OTWARTE ✅            │
│  Wejście B (wlan0) OTWARTE ✅           │
│  Wejście C (lo) OTWARTE ✅              │
│  <- wszyscy mogą wejść!                 │
│                                         │
│  --host 192.168.1.10 (konkretny IP)     │
│  Wejście A (eth0) OTWARTE ✅            │
│  ┌──────┐                               │
│  │  🚪  │ <- tylko przez to wejście!    │
│  └──────┘                               │
│  Wejście B (wlan0) ZAMKNIĘTE ❌         │
│  Wejście C (lo) ZAMKNIĘTE ❌            │
└─────────────────────────────────────────┘
```

## FastAPI + uvicorn - praktyczne przykłady

### Przykładowa aplikacja FastAPI

In [ ]:
# app.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "Hello from FastAPI!"}

@app.get("/info")
def info():
    import socket
    hostname = socket.gethostname()
    return {
        "hostname": hostname,
        "message": "Server is running!"
    }

### 1. Nasłuchiwanie TYLKO na localhost (127.0.0.1)

**Komenda:**
```bash
uvicorn app:app --host 127.0.0.1 --port 8000
```

**Co się dzieje?**
- Serwer nasłuchuje **tylko na interfejsie `lo` (loopback)**
- Dostępny **tylko z tego samego komputera**
- Sieć lokalna **NIE MA** dostępu
- Internet **NIE MA** dostępu

**Jak testować?**
```bash
# ✅ Działa - localhost
curl http://localhost:8000
curl http://127.0.0.1:8000

# ❌ NIE działa - IP interfejsu eth0 (192.168.1.10)
curl http://192.168.1.10:8000
# Connection refused!

# ❌ NIE działa - z innego komputera w sieci
# (z komputera kolegi w tej samej WiFi)
curl http://192.168.1.10:8000
# Connection refused!
```

**Kiedy używać?**
- ✅ Development lokalny
- ✅ Testowanie na swoim komputerze
- ✅ Bezpieczeństwo (nikt z sieci nie ma dostępu)

**Wyjście w terminalu:**
```
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
                              └─────┬─────┘
                           Tylko localhost!
```

### 2. Nasłuchiwanie na WSZYSTKICH interfejsach (0.0.0.0)

**Komenda:**
```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

**Co się dzieje?**
- Serwer nasłuchuje **na wszystkich interfejsach** (`lo`, `eth0`, `wlan0`, `docker0`)
- Dostępny z **localhost** (`127.0.0.1`)
- Dostępny z **sieci lokalnej** (np. `192.168.1.10` przez `eth0`)
- Dostępny z **internetu** (jeśli masz publiczny IP i brak firewall)
- Dostępny z **kontenerów Docker** (przez `docker0`)

**Jak testować?**
```bash
# ✅ Działa - localhost
curl http://localhost:8000
curl http://127.0.0.1:8000

# ✅ Działa - IP interfejsu eth0 (sieć lokalna)
curl http://192.168.1.10:8000

# ✅ Działa - IP interfejsu wlan0 (WiFi)
curl http://192.168.1.20:8000

# ✅ Działa - z innego komputera w sieci lokalnej
# (z komputera kolegi w tej samej WiFi)
curl http://192.168.1.10:8000

# ✅ Działa - z kontenera Docker
docker run --rm curlimages/curl http://172.17.0.1:8000
```

**Kiedy używać?**
- ✅ **Docker** (kontenery muszą mieć dostęp)
- ✅ Produkcja z firewall/reverse proxy (nginx/traefik filtruje ruch)
- ✅ Testowanie z innych urządzeń w sieci lokalnej (telefon, tablet)
- ⚠️ **UWAGA:** Bez firewall = każdy w sieci/internecie ma dostęp!

**Wyjście w terminalu:**
```
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
                              └────┬────┘
                        Wszystkie interfejsy!
```

**WAŻNE - co oznacza `0.0.0.0`?**

`0.0.0.0` to **specjalny adres** oznaczający "wszystkie dostępne interfejsy":
- `127.0.0.1` (lo - localhost)
- `192.168.1.10` (eth0 - kabel)
- `192.168.1.20` (wlan0 - WiFi)
- `172.17.0.1` (docker0 - Docker)

**NIE możesz** wejść na `http://0.0.0.0:8000` w przeglądarce!  
Musisz użyć **konkretnego IP** lub `localhost`.

### 3. Nasłuchiwanie na KONKRETNYM interfejsie (np. 192.168.1.10)

**Komenda:**
```bash
# Najpierw sprawdź swój IP interfejsu eth0
ip addr show eth0 | grep 'inet '
# Przykład: inet 192.168.1.10/24

# Uruchom serwer na konkretnym IP
uvicorn app:app --host 192.168.1.10 --port 8000
```

**Co się dzieje?**
- Serwer nasłuchuje **tylko na interfejsie `eth0`** (kabel sieciowy)
- Dostępny **tylko przez ten interfejs** (`192.168.1.10`)
- **NIE** dostępny przez localhost (`127.0.0.1`)
- **NIE** dostępny przez WiFi (`wlan0`)
- **NIE** dostępny przez Docker (`docker0`)

**Jak testować?**
```bash
# ❌ NIE działa - localhost
curl http://localhost:8000
curl http://127.0.0.1:8000
# Connection refused!

# ✅ Działa - konkretny IP interfejsu eth0
curl http://192.168.1.10:8000

# ❌ NIE działa - IP interfejsu wlan0 (jeśli różny)
curl http://192.168.1.20:8000
# Connection refused!

# ✅ Działa - z innego komputera w sieci (przez eth0)
curl http://192.168.1.10:8000
```

**Kiedy używać?**
- ✅ Ograniczenie dostępu do **jednej sieci** (np. tylko kabel, nie WiFi)
- ✅ Bezpieczeństwo - **nie eksponujesz** przez wszystkie interfejsy
- ⚠️ Rzadko używane w praktyce (częściej `127.0.0.1` lub `0.0.0.0` + firewall)

**Wyjście w terminalu:**
```
INFO:     Uvicorn running on http://192.168.1.10:8000 (Press CTRL+C to quit)
                              └──────┬───────┘
                         Tylko ten interfejs!
```

## Port vs Host - co to zmienia?

### --host (interfejs/IP)

**Decyduje:** **Kto** może się połączyć (z jakiego interfejsu/sieci)

```bash
--host 127.0.0.1  # Tylko localhost
--host 0.0.0.0    # Wszyscy (wszystkie interfejsy)
--host 192.168.1.10  # Tylko przez ten interfejs
```

### --port (numer portu)

**Decyduje:** **Pod jakim numerem** aplikacja nasłuchuje ("numer mieszkania")

```bash
--port 8000  # http://...:8000
--port 80    # http://.../ (domyślny HTTP, nie trzeba wpisywać :80)
--port 443   # https://.../ (domyślny HTTPS, nie trzeba wpisywać :443)
--port 3000  # http://...:3000
```

### Kombinacje:

```bash
# Localhost, port 8000
uvicorn app:app --host 127.0.0.1 --port 8000
# Dostępny: http://localhost:8000

# Wszystkie interfejsy, port 80
uvicorn app:app --host 0.0.0.0 --port 80
# Dostępny: http://localhost/, http://192.168.1.10/, etc.
# (Wymaga sudo/root - porty < 1024)

# Konkretny IP, port 3000
uvicorn app:app --host 192.168.1.10 --port 3000
# Dostępny: http://192.168.1.10:3000
```

## Praktyczne scenariusze - kiedy używać którego?

### 1. Development lokalny

```bash
uvicorn app:app --host 127.0.0.1 --port 8000
# lub krócej:
uvicorn app:app  # domyślnie 127.0.0.1:8000
```

**Dlaczego?**
- ✅ Bezpieczeństwo - nikt z sieci nie ma dostępu
- ✅ Szybkie - nie trzeba konfigurować firewall
- ✅ Testowanie tylko na swoim komputerze

### 2. Docker container

```bash
# W kontenerze MUSISZ użyć 0.0.0.0
uvicorn app:app --host 0.0.0.0 --port 8000
```

**Dlaczego?**
- ✅ Kontener ma swój własny network namespace
- ✅ `127.0.0.1` w kontenerze to **tylko wewnątrz kontenera**
- ✅ `0.0.0.0` pozwala na dostęp z hosta (przez mapowanie portów)

**Dockerfile:**
```dockerfile
FROM python:3.12
WORKDIR /app
COPY . .
RUN pip install fastapi uvicorn

# WAŻNE: --host 0.0.0.0 w kontenerze!
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Uruchomienie:**
```bash
docker build -t myapp .
docker run -p 8000:8000 myapp
#          └────┬────┘
#      Port hosta:Port kontenera

# Teraz dostępne na:
curl http://localhost:8000  # ✅ z hosta
```

### 3. Produkcja z reverse proxy (nginx/traefik)

W produkcji używamy **reverse proxy** (nginx, traefik), który obsługuje HTTPS, load balancing, caching.

**WAŻNE:** Konfiguracja FastAPI i nginx zależy od tego, czy są na **tym samym serwerze** czy **różnych serwerach**!

#### **Scenariusz A: Nginx i FastAPI na TYM SAMYM serwerze** (najczęstsze)

```bash
# FastAPI nasłuchuje TYLKO na localhost (bezpieczniejsze!)
uvicorn app:app --host 127.0.0.1 --port 8000
```

**Nginx config:**
```nginx
server {
    listen 80;
    server_name example.com;

    location / {
        # Przekieruj do FastAPI (localhost:8000)
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
```

**Dlaczego `127.0.0.1` dla FastAPI?**
- ✅ **Bezpieczeństwo** - FastAPI dostępne TYLKO dla nginx (nie dla internetu bezpośrednio)
- ✅ Nginx działa na tym samym serwerze, więc może się połączyć przez localhost
- ✅ Nie ma potrzeby wystawiać FastAPI na wszystkie interfejsy

**Schemat:**
```
Internet → Nginx (0.0.0.0:80) → FastAPI (127.0.0.1:8000)
             ↑                      ↑
       Publiczny dostęp      Tylko localhost
```

#### **Scenariusz B: Nginx i FastAPI na RÓŻNYCH serwerach**

```bash
# FastAPI nasłuchuje na wszystkich interfejsach (musi być dostępne z sieci)
uvicorn app:app --host 0.0.0.0 --port 8000
```

**Nginx config (na serwerze #1):**
```nginx
server {
    listen 80;
    server_name example.com;

    location / {
        # Przekieruj do FastAPI na INNYM serwerze
        proxy_pass http://192.168.1.10:8000;  # IP serwera FastAPI
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
```

**Dlaczego `0.0.0.0` dla FastAPI?**
- ✅ FastAPI musi być dostępne z **sieci** (inny serwer)
- ✅ Firewall na serwerze FastAPI powinien **zezwalać tylko IP nginx**

**Schemat:**
```
Internet → Nginx (Serwer #1, 0.0.0.0:80) → FastAPI (Serwer #2, 0.0.0.0:8000)
                                  └─────────────┘
                               Połączenie przez sieć lokalną
```

**Firewall na serwerze FastAPI:**
```bash
# Zezwól TYLKO nginx (192.168.1.5) na port 8000
sudo ufw allow from 192.168.1.5 to any port 8000
sudo ufw deny 8000  # Blokuj resztę
```

#### **Scenariusz C: Nginx na hoście, FastAPI w Docker**

```bash
# FastAPI w kontenerze - MUSI użyć 0.0.0.0
# (w Dockerfile lub docker-compose.yml)
uvicorn app:app --host 0.0.0.0 --port 8000
```

**Docker run z mapowaniem portu:**
```bash
# Mapuj port TYLKO na localhost hosta (bezpieczniejsze)
docker run -p 127.0.0.1:8000:8000 myapp
#             └─────┬─────┘
#          Tylko localhost hosta!
```

**Nginx config (na hoście):**
```nginx
server {
    listen 80;
    server_name example.com;

    location / {
        # Przekieruj do FastAPI w kontenerze (przez localhost hosta)
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
```

**Docker Compose (alternatywa):**
```yaml
services:
  fastapi:
    build: .
    # BEZ mapowania portów na hosta (nginx użyje nazwy serwisu)
    expose:
      - "8000"
  
  nginx:
    image: nginx
    ports:
      - "80:80"
    depends_on:
      - fastapi
```

**Nginx config (z docker-compose):**
```nginx
location / {
    # Użyj nazwy serwisu z docker-compose.yml
    proxy_pass http://fastapi:8000;
}
```

**Dlaczego `0.0.0.0` w kontenerze?**
- ✅ Kontener ma własny network namespace
- ✅ `127.0.0.1` w kontenerze to tylko **wewnątrz kontenera**
- ✅ `0.0.0.0` pozwala na połączenie z hosta/innych kontenerów

**Schemat:**
```
Internet → Nginx (host, 0.0.0.0:80) → FastAPI (container, 0.0.0.0:8000)
                                        └────┬────┘
                                  Dostęp przez Docker network
```

### Podsumowanie scenariuszy produkcyjnych:

| Scenariusz | FastAPI --host | Nginx proxy_pass | Kiedy używać? |
|------------|---------------|------------------|---------------|
| **A: Same server** | `127.0.0.1` | `http://127.0.0.1:8000` | Nginx i FastAPI na tym samym serwerze (najczęstsze) |
| **B: Different servers** | `0.0.0.0` | `http://192.168.1.10:8000` | Nginx i FastAPI na różnych serwerach |
| **C: Docker** | `0.0.0.0` | `http://127.0.0.1:8000` lub `http://fastapi:8000` | FastAPI w kontenerze, nginx na hoście/innym kontenerze |

**Kluczowa zasada:**
- Jeśli nginx i FastAPI **komunikują się przez localhost** → FastAPI używa `127.0.0.1` (bezpieczniejsze)
- Jeśli nginx i FastAPI **komunikują się przez sieć** → FastAPI używa `0.0.0.0` (+ firewall!)

### 4. Testowanie z telefonu/tabletu w tej samej WiFi

```bash
# Sprawdź swój IP w sieci WiFi
ip addr show wlan0 | grep 'inet '
# Przykład: inet 192.168.1.50/24

# Uruchom serwer na wszystkich interfejsach
uvicorn app:app --host 0.0.0.0 --port 8000

# Teraz możesz wejść z telefonu:
# http://192.168.1.50:8000
```

**Dlaczego?**
- ✅ Testowanie aplikacji mobilnych/responsywności
- ✅ Demo dla kolegów w tej samej sieci
- ⚠️ Pamiętaj wyłączyć po testach (bezpieczeństwo!)

## Jak sprawdzić kto może się połączyć?

### 1. Sprawdź na jakim porcie/interfejsie nasłuchuje aplikacja

```bash
# Linux - netstat
netstat -tuln | grep 8000

# Linux - ss (nowoczesne)
ss -tuln | grep 8000

# macOS - lsof
lsof -i :8000
```

**Przykładowe wyjście:**
```
# Nasłuchuje TYLKO na localhost
tcp  0  0  127.0.0.1:8000  0.0.0.0:*  LISTEN
           └─────┬─────┘
            Tylko localhost!

# Nasłuchuje na WSZYSTKICH interfejsach
tcp  0  0  0.0.0.0:8000    0.0.0.0:*  LISTEN
           └───┬───┘
         Wszystkie interfejsy!

# Nasłuchuje na KONKRETNYM IP
tcp  0  0  192.168.1.10:8000  0.0.0.0:*  LISTEN
           └──────┬───────┘
          Tylko ten interfejs!
```

### 2. Testuj połączenia

```bash
# Test z localhost
curl http://localhost:8000

# Test z konkretnego IP
curl http://192.168.1.10:8000

# Test z innego komputera w sieci
# (zaloguj się SSH lub fizycznie)
curl http://192.168.1.10:8000
```

## ⚠️ Bezpieczeństwo - WAŻNE!

### Zagrożenia przy `--host 0.0.0.0`

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

**Kto ma dostęp?**
- ✅ Ty (localhost)
- ✅ Koledzy w tej samej sieci WiFi/LAN
- ⚠️ Każdy w internecie (jeśli masz publiczny IP i brak firewall)
- ⚠️ Złośliwe osoby mogą zeskanować porty i znaleźć Twoją aplikację

### Dobre praktyki:

1. **Development lokalny:** Używaj `127.0.0.1`
   ```bash
   uvicorn app:app --host 127.0.0.1 --port 8000
   ```

2. **Docker:** Używaj `0.0.0.0` w kontenerze, ale **NIE** publikuj portów publicznie
   ```bash
   # Dobrze - port dostępny tylko na localhost hosta
   docker run -p 127.0.0.1:8000:8000 myapp
   
   # Źle - port dostępny dla każdego!
   docker run -p 8000:8000 myapp
   ```

3. **Produkcja:** Używaj `0.0.0.0` + **reverse proxy** (nginx/traefik) + **firewall**
   - Firewall blokuje bezpośredni dostęp do portu 8000
   - Tylko nginx (port 80/443) jest dostępny z internetu
   - Nginx przekierowuje do FastAPI (localhost:8000)

4. **Testowanie z telefonu:** Używaj `0.0.0.0`, ale **tylko tymczasowo**
   ```bash
   # Włącz na chwilę testów
   uvicorn app:app --host 0.0.0.0 --port 8000
   
   # WYŁĄCZ po testach!
   # (Ctrl+C)
   ```

### Firewall (Linux):

```bash
# Zablokuj port 8000 z internetu (zostaw dla sieci lokalnej)
sudo ufw allow from 192.168.1.0/24 to any port 8000
sudo ufw deny 8000
```

## 📝 Podsumowanie

### Kluczowe pojęcia:

| Termin | Analogia | Przykład |
|--------|----------|----------|
| **Interfejs sieciowy** | Drzwi wejściowe do bloku | `eth0`, `wlan0`, `lo` |
| **IP Address** | Adres ulicy dla danego wejścia | `192.168.1.10`, `127.0.0.1` |
| **Port** | Numer mieszkania w bloku | `80`, `443`, `8000` |
| **Socket** | Pełny adres (ulica + mieszkanie) | `192.168.1.10:8000` |

### Opcje nasłuchiwania:

| --host | Kto ma dostęp? | Kiedy używać? |
|--------|----------------|---------------|
| `127.0.0.1` | Tylko localhost | Development lokalny |
| `0.0.0.0` | Wszyscy (wszystkie interfejsy) | Docker, produkcja + firewall |
| Konkretny IP | Tylko przez ten interfejs | Ograniczenie do jednej sieci (rzadko) |

### Praktyczne komendy:

```bash
# Development - tylko localhost
uvicorn app:app --host 127.0.0.1 --port 8000

# Docker - wszystkie interfejsy w kontenerze
uvicorn app:app --host 0.0.0.0 --port 8000

# Produkcja - wszystkie interfejsy + reverse proxy
uvicorn app:app --host 0.0.0.0 --port 8000
# (+ nginx/traefik na porcie 80/443)

# Sprawdź na jakim interfejsie nasłuchuje
ss -tuln | grep 8000
netstat -tuln | grep 8000

# Sprawdź swoje interfejsy i IP
ip addr
ip -br addr  # krótka wersja
```

### Najważniejsze zasady:

1. **Interfejs ≠ Port** - to dwa różne pojęcia!
2. **`127.0.0.1`** = bezpieczne (tylko localhost)
3. **`0.0.0.0`** = wszyscy mają dostęp (używaj z firewall!)
4. **Docker** zawsze wymaga `0.0.0.0` w kontenerze
5. **Produkcja** = `0.0.0.0` + reverse proxy + firewall

### Błędy do uniknięcia:

❌ **"Interfejs to port 8000"** - interfejs to `eth0`, port to `8000`  
❌ **"127.0.0.1 to interfejs"** - `lo` to interfejs, `127.0.0.1` to jego IP  
❌ **"0.0.0.0 to IP"** - to specjalny adres "wszystkie interfejsy", nie konkretny IP  
❌ **Development z `0.0.0.0` bez firewall** - każdy w sieci ma dostęp!  
❌ **Docker z `127.0.0.1`** - kontener nie będzie dostępny z hosta!

---

## ❓ FAQ - Najczęstsze pytania

### Q: Czy interfejsy mają różne adresy IP? Myślałem, że komputer ma jeden adres IP, a nie tyle ile ma interfejsów!

**A: TAK! Każdy interfejs ma swój własny adres IP.**

**Komputer ma TYLE adresów IP ile ma aktywnych interfejsów.**

**Praktyczny przykład:**

```bash
# Sprawdź adresy IP na swoim komputerze
ip addr

# Wynik (przykładowy):
1: lo: <LOOPBACK,UP>
    inet 127.0.0.1/8         ← Adres IP #1 (loopback)

2: eth0: <BROADCAST,MULTICAST,UP>
    inet 192.168.1.10/24     ← Adres IP #2 (kabel)

3: wlan0: <BROADCAST,MULTICAST,UP>
    inet 192.168.1.20/24     ← Adres IP #3 (WiFi)

4: docker0: <BROADCAST,MULTICAST>
    inet 172.17.0.1/16       ← Adres IP #4 (Docker)
```

**Ten sam komputer ma 4 różne adresy IP jednocześnie!**

**Dlaczego?**

Ponieważ możesz być podłączony do **różnych sieci jednocześnie**:
- **Kabel (eth0)** → sieć `192.168.1.0/24` → adres `192.168.1.10`
- **WiFi (wlan0)** → sieć `192.168.1.0/24` → adres `192.168.1.20` (może być inna sieć!)
- **Loopback (lo)** → zawsze `127.0.0.1` (localhost)
- **Docker (docker0)** → sieć kontenerów `172.17.0.0/16` → adres `172.17.0.1`

**Co to oznacza dla FastAPI?**

Jeśli uruchomisz serwer z `--host 0.0.0.0`:

```bash
uvicorn app:app --host 0.0.0.0 --port 8000

# Serwer jest dostępny pod WSZYSTKIMI adresami IP:
curl http://127.0.0.1:8000       # ✅ przez loopback
curl http://192.168.1.10:8000    # ✅ przez kabel
curl http://192.168.1.20:8000    # ✅ przez WiFi
curl http://172.17.0.1:8000      # ✅ przez Docker
```

**To TEN SAM serwer, ale dostępny pod 4 różnymi adresami!**

**Kiedy mówimy "komputer ma jeden adres IP"?**

To uproszczenie! Zazwyczaj myślimy o:
1. **Publicznym IP z internetu** - router NAT pokazuje jeden IP na zewnątrz (np. `89.64.123.45`)
2. **Głównym IP w sieci lokalnej** - najczęściej używany interfejs (eth0 lub wlan0)

Ale **technicznie każdy interfejs ma swój własny adres IP**.

---

### Q: Jak działa `fastapi dev`? Na jakim interfejsie i porcie się uruchamia?

**A: `fastapi dev` domyślnie uruchamia się na `127.0.0.1:8000` (localhost) z auto-reload.**

**Domyślne ustawienia:**

```bash
fastapi dev app.py

# To jest równoważne:
uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```

- **Host:** `127.0.0.1` (tylko localhost) - bezpieczne dla development
- **Port:** `8000`
- **Auto-reload:** włączony (automatycznie przeładowuje po zmianie kodu)

**Dostępny tylko z localhost:**

```bash
# ✅ Działa
curl http://localhost:8000
curl http://127.0.0.1:8000

# ❌ NIE działa z sieci lokalnej
curl http://192.168.1.10:8000  # Connection refused
```

**Jak zmienić host/port?**

```bash
# Zmień host na 0.0.0.0 (wszystkie interfejsy)
fastapi dev app.py --host 0.0.0.0

# Zmień port na 3000
fastapi dev app.py --port 3000

# Zmień obie wartości
fastapi dev app.py --host 0.0.0.0 --port 3000
```

**`fastapi dev` vs `fastapi run`**

| Polecenie | Host | Port | Auto-reload | Kiedy używać? |
|-----------|------|------|-------------|---------------|
| `fastapi dev` | `127.0.0.1` | 8000 | ✅ Włączony | Development lokalny |
| `fastapi run` | `0.0.0.0` | 8000 | ❌ Wyłączony | Produkcja (ale lepiej użyć gunicorn/uvicorn bezpośrednio) |

**Przykład dla Docker (MUSISZ użyć 0.0.0.0):**

```dockerfile
# ❌ Źle - w kontenerze nie będzie dostępny z hosta
CMD ["fastapi", "dev", "app.py"]

# ✅ Dobrze - dostępny z hosta
CMD ["fastapi", "dev", "app.py", "--host", "0.0.0.0"]

# ✅ Jeszcze lepiej dla produkcji
CMD ["fastapi", "run", "app.py", "--host", "0.0.0.0"]
```

**Podsumowanie:**
- `fastapi dev` = localhost tylko (`127.0.0.1:8000`) + auto-reload
- Dla Docker/testowania z sieci: dodaj `--host 0.0.0.0`
- Dla produkcji: użyj `fastapi run` lub bezpośrednio `uvicorn` z gunicorn

---

### Q: Dla `0.0.0.0` napisałeś "Dostępny z internetu (jeśli masz publiczny IP)". Chyba każdy ma publiczny adres IP? Przecież jak się łączę z czymś, to to coś widzi jaki mam adres, inaczej nie mogłoby odesłać odpowiedzi do mnie.

**A: NIE, większość użytkowników NIE MA publicznego IP na swoim komputerze!**

**Twój komputer ma PRYWATNY adres IP, a ROUTER ma publiczny IP.**

**Schemat:**

```
Internet widzi:         89.64.123.45 (router - PUBLICZNY)
                              ↓
                        ┌──────────┐
                        │  ROUTER  │
                        │   (NAT)  │
                        └──────────┘
                              ↓
Twoja sieć lokalna:    192.168.1.10 (komputer - PRYWATNY)
                       192.168.1.20 (telefon - PRYWATNY)
                       192.168.1.30 (laptop - PRYWATNY)
```

**Twój komputer ma PRYWATNY IP:**
- `192.168.1.10` (lub `10.x.x.x`, lub `172.16.x.x`)
- **NIE jest widoczny z internetu**
- Tylko widoczny w Twojej sieci lokalnej

**Router ma PUBLICZNY IP:**
- `89.64.123.45` (przykład)
- **Widoczny z internetu**
- To TEN adres widzą serwery zewnętrzne

**Jak odpowiedź wraca do Ciebie? (NAT - Network Address Translation)**

Router prowadzi **tabelę NAT**:

```
1. Ty wysyłasz request:
   Komputer (192.168.1.10:54321) → google.com:443
   
2. Router zmienia adres źródłowy i ZAPISUJE:
   Router → Internet: 89.64.123.45:54321 → google.com:443
   Router zapisuje: "Port 54321 = komputer 192.168.1.10:54321"
   
3. Google widzi:
   Request od: 89.64.123.45:54321 (NIE widzi Twojego 192.168.1.10!)
   
4. Google odpowiada:
   Odpowiedź do: 89.64.123.45:54321
   
5. Router sprawdza tabelę NAT:
   "Port 54321? Aha, to komputer 192.168.1.10:54321"
   Router przekierowuje → 192.168.1.10:54321
   
6. Ty dostajesz odpowiedź! ✅
```

**Router wie gdzie przekierować odpowiedź, bo SAM zapisał skąd wyszedł request!**

**Sprawdź czy masz publiczny IP bezpośrednio:**

```bash
# Twój prywatny IP (w sieci lokalnej)
ip addr show | grep "inet "
# Przykład: 192.168.1.10

# Publiczny IP routera (widoczny z internetu)
curl ifconfig.me
# Przykład: 89.64.123.45
```

Jeśli są różne - **NIE masz publicznego IP bezpośrednio**, masz NAT!

---

### Q: Rozumiem, że NAT wie na co przekierować ruch wychodzący, bo sobie zapisuje z jakiej maszyny wyszedł. Ale jak mamy ruch PRZYCHODZĄCY (nowy), to już nie wie i co? Blokuje?

**A: TAK! Router domyślnie BLOKUJE wszystkie NOWE połączenia przychodzące z internetu!**

**Ruch WYCHODZĄCY (outbound) - działa automatycznie:**

```
1. Ty inicjujesz połączenie:
   Komputer (192.168.1.10:54321) → google.com:443
   
2. Router ZAPISUJE w tabeli NAT:
   "Port 54321 = 192.168.1.10:54321"
   
3. Google odpowiada:
   google.com:443 → Router (89.64.123.45:54321)
   
4. Router sprawdza tabelę NAT:
   "Port 54321? Aha, to 192.168.1.10:54321"
   Router przekierowuje → 192.168.1.10:54321 ✅
```

**Router WIE gdzie przekierować odpowiedź!**

**Ruch PRZYCHODZĄCY (inbound) - NOWY request - BLOKOWANY!**

```
1. Ktoś z internetu próbuje połączyć się:
   Hacker → 89.64.123.45:8000 (Twój FastAPI)
   
2. Router dostaje request:
   "Ktoś chce połączyć się z portem 8000"
   
3. Router sprawdza tabelę NAT:
   "Port 8000? Nie ma wpisu w tabeli NAT!"
   (bo to NOWY request, nie odpowiedź na twój request)
   
4. Router:
   ❌ BLOKUJE/DROPUJE (firewall)
   Pakiet jest odrzucany
```

**Router NIE WIE gdzie przekierować → BLOKUJE!**

**Dlaczego router blokuje?**

**Bezpieczeństwo!** Router działa jak firewall - domyślnie:
- ✅ **Ruch wychodzący** (outbound) - POZWÓL (Ty inicjujesz)
- ❌ **Ruch przychodzący** (inbound) - BLOKUJ (chyba że skonfigurujesz Port Forwarding)

**Port Forwarding - ręczna reguła:**

Jeśli CHCESZ aby ruch z internetu dochodził do Twojego FastAPI:

```
Router config (Port Forwarding):
- Zewnętrzny port: 8000
- Przekieruj do IP: 192.168.1.10
- Wewnętrzny port: 8000

Teraz:
Ktoś z internetu → 89.64.123.45:8000
Router: "Mam regułę! Przekierowuję do 192.168.1.10:8000"
Router → 192.168.1.10:8000 ✅
```

**Podsumowanie:**

| Typ ruchu | Router wie gdzie przekierować? | Co się dzieje? |
|-----------|--------------------------------|----------------|
| **Ruch wychodzący** (Ty → Internet) | ✅ TAK (zapisuje w tabeli NAT) | Przekierowuje odpowiedź do Ciebie |
| **Odpowiedź** (Internet → Ty) | ✅ TAK (sprawdza tabelę NAT) | Przekierowuje do Ciebie |
| **Nowy ruch przychodzący** | ❌ NIE (brak wpisu w NAT) | ❌ BLOKUJE |
| **Nowy ruch + Port Forwarding** | ✅ TAK (statyczna reguła) | Przekierowuje według reguły |

---

### Q: Router musi zapisywać coś więcej niż tylko adres wychodzący! Bo załóżmy, że dwa urządzenia uderzają na ten sam adres (np. google.com). Skąd router miałby wtedy wiedzieć, zwrotka z tego adresu do której z maszyn ma trafić?

**A: Dokładnie! Router zapisuje nie tylko IP, ale SOCKET (IP + PORT)!**

**Przykład - dwa komputery łączą się z google.com:**

```
Komputer 1 (192.168.1.10) → google.com:443
Komputer 2 (192.168.1.20) → google.com:443

Oba idą do TEGO SAMEGO adresu! Jak router to rozróżni?
```

**Tabela NAT zapisuje SOCKET (IP:PORT):**

```
┌──────────────────────────────────────────────────────┐
│  TABELA NAT w routerze                               │
├────────────────────────┬─────────────────────────────┤
│ Wewnętrzny SOCKET      │ Zewnętrzny SOCKET           │
│ (prywatny IP:port)     │ (publiczny IP:port)         │
├────────────────────────┼─────────────────────────────┤
│ 192.168.1.10:54321     │ 89.64.123.45:54321          │
│ 192.168.1.20:54322     │ 89.64.123.45:54322          │
│ 192.168.1.10:54323     │ 89.64.123.45:54323          │
└────────────────────────┴─────────────────────────────┘
         ↑                        ↑
    IP + PORT!              IP + PORT!
```

**Kluczem jest PORT, nie tylko IP!**

**Krok po kroku:**

```
1. Komputer 1 wysyła request:
   Źródło: 192.168.1.10:54321 (losowy port)
   Cel: google.com:443
   Router zapisuje: "Port 54321 = 192.168.1.10:54321"

2. Komputer 2 wysyła request (do tego samego google.com!):
   Źródło: 192.168.1.20:54322 (INNY losowy port)
   Cel: google.com:443
   Router zapisuje: "Port 54322 = 192.168.1.20:54322"

3. Google odpowiada na PIERWSZY request:
   Cel: 89.64.123.45:54321
   Router sprawdza: "Port 54321? To 192.168.1.10:54321"
   Router przekierowuje → 192.168.1.10:54321 ✅

4. Google odpowiada na DRUGI request:
   Cel: 89.64.123.45:54322
   Router sprawdza: "Port 54322? To 192.168.1.20:54322"
   Router przekierowuje → 192.168.1.20:54322 ✅
```

**Router rozróżnia po PORCIE zewnętrznym (54321 vs 54322)!**

**Kluczowe punkty:**
- Router zapisuje cały **SOCKET (IP:PORT)**, nie sam IP
- **PORT źródłowy** to klucz w tabeli NAT
- Każde połączenie ma **unikalny port źródłowy** (losowo przypisany przez system)
- Router może obsługiwać **tysiące równoczesnych połączeń** (tyle ile portów: ~65000)

---

### Q: Dlaczego kontenery dostają losowe IP z zakresu przy każdym uruchomieniu? Dlaczego za każdym razem nie przypisywać im tych samych adresów?

**A: Bo kontenery są TYMCZASOWE i możesz mieć ich WIELE jednocześnie.**

**Praktyczny przykład - skalowanie:**

```
Masz serwis "fastapi-app":
- Instancja #1 (172.17.0.5)
- Instancja #2 (172.17.0.6)  ← skalowanie (2 kontenery jednocześnie!)
- Instancja #3 (172.17.0.7)  ← jeszcze więcej skalowania

Jedna instancja pada → Kubernetes uruchamia nową:
- Instancja #4 (172.17.0.8)  ← nowy kontener, nowe IP
```

**Gdyby były STAŁE IP - problemy:**

1. **Skalowanie poziome (wiele kopii)**
   - Chcesz uruchomić 10 instancji tego samego obrazu
   - ❌ Konflikt: wszystkie chcą tego samego IP!
   
2. **Rolling update (zero downtime)**
   ```
   Stara wersja (v1) działa na 172.17.0.5
   Uruchamiasz nową (v2)...
   ❌ Konflikt IP! v2 też chce 172.17.0.5
   Musisz czekać aż v1 się wyłączy
   = przestój!
   ```
   
3. **Automatyczne restarty**
   ```
   Kontener pada, Kubernetes restartuje...
   ❌ IP 172.17.0.5 zajęte (jeszcze czyszczenie starego)
   Musisz czekać aż IP się zwolni
   ```

4. **Marnowanie puli IP**
   - Musisz REZERWOWAĆ IP dla każdego potencjalnego kontenera
   - Masz 100 różnych serwisów × 10 instancji = 1000 zarezerwowanych IP
   - Większość czasu większość NIE działa (marnowanie)

**Z LOSOWYMI IP z puli - zalety:**

1. **Skalowanie działa automatycznie**
   ```
   Uruchamiasz 10 instancji:
   - Docker/K8s bierze 10 wolnych IP z puli (np. 172.17.0.5-14)
   - Żadnych konfliktów
   ```

2. **Rolling update bez przestoju**
   ```
   Stara wersja: 172.17.0.5
   Nowa wersja:  172.17.0.8 (nowe IP z puli)
   Oba działają równolegle! Load balancer przekierowuje ruch.
   Dopiero jak nowa działa → stara się wyłącza → IP wraca do puli
   ```

3. **Efektywne użycie IP**
   - Pula 256 IP (172.17.0.0/24)
   - W danym momencie używasz tylko 20 → reszta dostępna
   - Kontener pada → IP wraca do puli → można użyć ponownie

**Jak się łączysz skoro IP się zmienia?**

```yaml
# ❌ NIE łączysz się po IP!
proxy_pass http://172.17.0.5:8000;  # Za chwilę to IP będzie nieaktualne!

# ✅ Łączysz się po NAZWIE serwisu
proxy_pass http://fastapi-app:8000;  # Docker/K8s rozwiązuje DNS
                ↑
         Nazwa serwisu (stała)
         Docker/K8s automatycznie tłumaczy na aktualny IP
         + load balancing między instancjami
```

**Podsumowanie:**

| Stałe IP | Losowe IP z puli |
|----------|-----------------|
| ❌ Konflikt przy skalowaniu | ✅ Automatyczne skalowanie |
| ❌ Rolling update = przestój | ✅ Zero downtime |
| ❌ Marnowanie zarezerwowanych IP | ✅ Efektywne użycie puli |
| ❌ Ręczne zarządzanie | ✅ Automatyczne zarządzanie |

**Filozofia kontenerów:**
- Kontenery są **tymczasowe** (uruchamiane/kasowane ciągle)
- Nie dbasz o konkretny kontener (nie ma sentymentu)
- Dbasz o **serwis jako całość** (nazwa + wiele instancji)
- Infrastruktura (Docker/K8s) zarządza szczegółami (IP, routing)